# Huấn Luyện & Đánh Giá Mô Hình PhoBERT (UIT-VSFC Sentiment Analysis)
---
### Mục tiêu & Quy trình chuẩn phương pháp luận Machine Learning:
1. **Xử lý dữ liệu đồng bộ**: Áp dụng hàm `clean_text` chuẩn hóa teencode học thuật trên cả 3 tập Train, Validation, Test.
2. **Huấn luyện PhoBERT**: Tích hợp trọng số phạt lỗi `Class Weights` vào `CrossEntropyLoss` để xử lý mất cân bằng nhãn.
3. **Kiểm soát Overfitting**: Theo dõi Loss trên tập **Validation** qua từng Epoch (tuyệt đối không dùng tập Test khi huấn luyện).
4. **Tinh chỉnh ngưỡng quyết định (Threshold Tuning)**: Quét tìm ngưỡng tối ưu cho nhãn Trung tính trên tập **Validation**.
5. **Nghiệm thu độc lập trên tập Test (Unseen Data)**: Áp dụng ngưỡng tối ưu sang tập Test để lấy số liệu báo cáo chính thức.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix

# Thiết lập thiết bị phần cứng (Ưu tiên GPU CUDA nếu có)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị tính toán: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Hàm tự động tìm đường dẫn linh hoạt (hỗ trợ chạy từ mọi thư mục)
def resolve_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]

# Hàm tiền xử lý & chuẩn hóa teencode học thuật đồng bộ
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\b(gv|thầy|cô)\b", "giảng viên", text)
    text = re.sub(r"\b(sv)\b", "sinh viên", text)
    text = re.sub(r"\b(slide|slđ)\b", "bài giảng", text)
    text = re.sub(r"\b(đc)\b", "được", text)
    text = re.sub(r"\b(khg|ko|kh|k)\b", "không", text)
    text = re.sub(r"\b(ok|ổn)\b", "tạm ổn", text)
    text = re.sub(r"[^\w\s,.-+]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Lớp cấu trúc PyTorch Dataset cho PhoBERT
class SentimentDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.labels = labels
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Định nghĩa cấu trúc hàm và Dataset thành công!")


In [ ]:
# Nạp dữ liệu Train, Validation và Test
TRAIN_PATH = resolve_path(["../../05_Data/processed/train_cleaned.csv", "../05_Data/processed/train_cleaned.csv", "05_Data/processed/train_cleaned.csv"])
VAL_PATH   = resolve_path(["../../05_Data/raw/validation.csv", "../05_Data/raw/validation.csv", "05_Data/raw/validation.csv"])
TEST_PATH  = resolve_path(["../../05_Data/raw/test.csv", "../05_Data/raw/test.csv", "05_Data/raw/test.csv"])

train_df = pd.read_csv(TRAIN_PATH).dropna()
val_df   = pd.read_csv(VAL_PATH).dropna()
test_df  = pd.read_csv(TEST_PATH).dropna()

# Áp dụng clean_text đồng bộ cho cả tập Validation và Test
val_df["cleaned_comment"]  = val_df["comment"].apply(clean_text)
test_df["cleaned_comment"] = test_df["comment"].apply(clean_text)

# Khởi tạo Tokenizer PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
MAX_LENGTH = 256

train_dataset = SentimentDataset(
    texts=train_df["tokenized"].values,
    labels=train_df["label"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = SentimentDataset(
    texts=val_df["cleaned_comment"].values,
    labels=val_df["label"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = SentimentDataset(
    texts=test_df["cleaned_comment"].values,
    labels=test_df["label"].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("Nạp dữ liệu 3 tập thành công:")
print(f" - Tập Train:      {len(train_dataset):,} mẫu")
print(f" - Tập Validation: {len(val_dataset):,} mẫu")
print(f" - Tập Test:       {len(test_dataset):,} mẫu")


In [ ]:
# Cấu hình DataLoader
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_EVAL  = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False)

CHECKPOINT_DIR = resolve_path(["../../06_Model_Checkpoints", "../06_Model_Checkpoints", "06_Model_Checkpoints"])
has_checkpoint = os.path.exists(os.path.join(CHECKPOINT_DIR, "model.safetensors"))

if has_checkpoint:
    print(f"Phát hiện checkpoint tối ưu có sẵn tại: {CHECKPOINT_DIR}")
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
else:
    print("Khởi tạo PhoBERT nguyên bản từ vinai/phobert-base...")
    model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=3)

model.to(DEVICE)
print("Khởi tạo mô hình và chuyển lên thiết bị hoàn tất.")


In [ ]:
# 1. Tính toán Class Weights từ phân bố nhãn tập Train
train_labels = train_df["label"].values
classes, counts = np.unique(train_labels, return_counts=True)
total_samples = len(train_labels)
class_weights = total_samples / (len(classes) * counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print(f"Phân bố nhãn tập Train: {dict(zip(classes, counts))}")
print(f"Mảng Class Weights tối ưu: {class_weights}")

# 2. Khởi tạo Hàm mất mát và Optimizer
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
EPOCHS = 5
LEARNING_RATE = 1e-5
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)


In [ ]:
# CỜ ĐIỀU KHIỂN HUẤN LUYỆN:
# - Đặt SKIP_TRAINING = True nếu bạn muốn sử dụng checkpoint đã train có sẵn trong 06_Model_Checkpoints để đánh giá nhanh.
# - Đặt SKIP_TRAINING = False nếu bạn muốn huấn luyện lại 5 epochs từ đầu.
SKIP_TRAINING = False if has_checkpoint else False

train_loss_history = []
val_loss_history = []
RESULTS_DIR = resolve_path(["../../04_Results", "../04_Results", "04_Results"])
os.makedirs(RESULTS_DIR, exist_ok=True)

if not SKIP_TRAINING:
    print("=== BẮT ĐẦU HUẤN LUYỆN VÀ THEO DÕI TRÊN TẬP VALIDATION ===")
    for epoch in range(EPOCHS): 
        print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")
        
        # Phase 1: Train
        model.train()
        total_train_loss = 0
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_train_loss += loss.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            if step % 50 == 0 and step > 0:
                print(f"  Bước {step}/{len(train_loader)} | Loss hiện tại: {loss.item():.4f}")
                
        avg_train_loss = total_train_loss / len(train_loader)
        train_loss_history.append(avg_train_loss)
        print(f"Trung bình Train Loss: {avg_train_loss:.4f}")
        
        # Phase 2: Validation (Theo dõi chuẩn trên val_loader - Tuyệt đối không dùng test_loader!)
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits, labels)
                total_val_loss += loss.item()
                
        avg_val_loss = total_val_loss / len(val_loader)
        val_loss_history.append(avg_val_loss)
        print(f"Trung bình Validation Loss: {avg_val_loss:.4f}")
        
    # Lưu checkpoint
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    model.save_pretrained(CHECKPOINT_DIR)
    tokenizer.save_pretrained(CHECKPOINT_DIR)
    print(f"Đã lưu checkpoint mô hình tại: {CHECKPOINT_DIR}")
    
    # Vẽ biểu đồ Loss Curve Train vs Validation
    plt.figure(figsize=(9, 5))
    plt.plot(range(1, EPOCHS + 1), train_loss_history, label="Loss Tập Huấn Luyện (Train Set)", marker="o")
    plt.plot(range(1, EPOCHS + 1), val_loss_history, label="Loss Tập Đánh Giá (Validation Set)", marker="s")
    plt.title("Đường cong hội tụ kiểm soát Overfitting (Train Loss vs Validation Loss)")
    plt.xlabel("Epochs")
    plt.ylabel("Loss Value")
    plt.legend()
    plt.grid(True)
    loss_path = os.path.join(RESULTS_DIR, "loss_curves.png")
    plt.savefig(loss_path, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu biểu đồ loss tại: {loss_path}")
else:
    print("SKIP_TRAINING = True: Bỏ qua quá trình train, nạp thẳng checkpoint có sẵn để tiến hành đánh giá.")


In [ ]:
print("=== BƯỚC 1: TINH CHỈNH NGƯỠNG TỐI ƯU TRÊN TẬP VALIDATION (CHUẨN ML) ===")

model.eval()
val_logits_list = []
val_labels_list = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        val_logits_list.extend(outputs.logits.cpu().numpy())
        val_labels_list.extend(labels.cpu().numpy())

val_logits = np.array(val_logits_list)
y_val_true = np.array(val_labels_list)

# Tính toán Softmax xác suất mềm [0, 1]
exp_logits = np.exp(val_logits - np.max(val_logits, axis=-1, keepdims=True))
val_probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

# Quét tìm ngưỡng tối ưu cho nhãn 1 (Trung tính) trên tập Validation
best_threshold = 0.5
best_val_macro_f1 = 0.0

for threshold in np.arange(0.18, 0.50, 0.01):
    preds = []
    for prob in val_probs:
        if prob[1] >= threshold:
            preds.append(1)
        else:
            preds.append(0 if prob[0] > prob[2] else 2)
            
    rep = classification_report(y_val_true, preds, output_dict=True, zero_division=0)
    macro_f1 = rep["macro avg"]["f1-score"]
    
    if macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = macro_f1
        best_threshold = round(float(threshold), 2)

print(f"\n[KẾT QUẢ TỐI ƯU HÓA NGƯỠNG TRÊN TẬP VALIDATION]")
print(f" -> Ngưỡng quyết định tối ưu cho nhãn Trung tính: {best_threshold}")
print(f" -> Macro-F1 tối đa trên tập Validation:          {best_val_macro_f1:.4f}")


In [ ]:
print(f"=== BƯỚC 2: NGHIỆM THU ĐỘC LẬP TRÊN TẬP TEST (ÁP DỤNG NGƯỠNG {best_threshold}) ===")

test_logits_list = []
test_labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        test_logits_list.extend(outputs.logits.cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_logits = np.array(test_logits_list)
y_test_true = np.array(test_labels_list)

# Tính toán Softmax xác suất mềm trên tập Test
exp_test = np.exp(test_logits - np.max(test_logits, axis=-1, keepdims=True))
test_probs = exp_test / np.sum(exp_test, axis=-1, keepdims=True)

# Áp dụng ngưỡng tối ưu (đã chốt từ tập Validation) sang tập Test
final_test_preds = []
for prob in test_probs:
    if prob[1] >= best_threshold:
        final_test_preds.append(1)
    else:
        final_test_preds.append(0 if prob[0] > prob[2] else 2)

final_test_preds = np.array(final_test_preds)

# 1. In Báo Cáo Hiệu Năng Phân Loại Chính Thức (Classification Report)
target_names = ["Tiêu cực (0)", "Trung tính (1)", "Tích cực (2)"]
print("\nBÁO CÁO KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP TEST:")
print(classification_report(y_test_true, final_test_preds, target_names=target_names, digits=4))

# 2. Xuất bảng dữ liệu metric ra file final_metrics.csv
report_dict = classification_report(y_test_true, final_test_preds, target_names=target_names, output_dict=True)
df_metrics = pd.DataFrame(report_dict).transpose()
metrics_path = os.path.join(RESULTS_DIR, "final_metrics.csv")
df_metrics.to_csv(metrics_path, index=True)
print(f"Đã cập nhật bảng điểm chính thức tại: {metrics_path}")

# 3. Vẽ và lưu Ma trận nhầm lẫn tối hậu (Confusion Matrix)
cm = confusion_matrix(y_test_true, final_test_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Dự đoán 0", "Dự đoán 1", "Dự đoán 2"],
            yticklabels=["Thực tế 0", "Thực tế 1", "Thực tế 2"])
plt.title(f"Ma Trận Nhầm Lẫn Tối Hậu Trên Tập Test (Ngưỡng = {best_threshold})")
plt.ylabel("Nhãn Thực Tế")
plt.xlabel("Nhãn Dự Đoán")
cm_path = os.path.join(RESULTS_DIR, "phobert_confusion_matrix.png")
plt.savefig(cm_path, bbox_inches="tight")
plt.show()
print(f"Đã cập nhật ma trận nhầm lẫn tại: {cm_path}")

# 4. Xuất danh sách mẫu lỗi phục vụ phân tích (Error Analysis)
test_raw_comments = test_df["comment"].values
error_indices = np.where(final_test_preds != y_test_true)[0]

error_list = []
for idx in error_indices:
    error_list.append({
        "comment": test_raw_comments[idx],
        "true_label": y_test_true[idx],
        "pred_label": final_test_preds[idx],
        "prob_0": round(float(test_probs[idx][0]), 4),
        "prob_1": round(float(test_probs[idx][1]), 4),
        "prob_2": round(float(test_probs[idx][2]), 4)
    })

df_error = pd.DataFrame(error_list)
error_path = os.path.join(RESULTS_DIR, "error_analysis.csv")
df_error.to_csv(error_path, index=False, encoding="utf-8-sig")
print(f"Đã xuất {len(df_error):,} mẫu dự đoán sai vào: {error_path}")
